# Day 9 Microservices Architecture Patterns
|                Topic               |                          Short Summary                         |
|:----------------------------------:|:---------------------------------------------------------------|
|            Microservices           | Small independent services for specific business functions.    |
|             API Gateway            | Single entry point that routes requests to services.           |
|        Database per Service        | Each service owns its own database.                            |
|      Event-Driven Architecture     | Services communicate through asynchronous events.              |
|                Kafka               | Event broker that transports and stores messages.              |
|              Producer              | Sends events to Kafka.                                         |
|                Topic               | Category where Kafka events are stored.                        |
|              Consumer              | Reads and processes Kafka events.                              |
|           Circuit Breaker          | Stops requests to failing services.                            |
|            Closed State            | Requests flow normally.                                        |
|             Open State             | Requests are blocked.                                          |
|           Half-Open State          | Tests if a service has recovered.                              |
|            Retry Pattern           | Retries temporary failures automatically.                      |
|          Bulkhead Pattern          | Isolates failures to protect other services.                   |
| Retry + Circuit Breaker + Bulkhead | Combined patterns for resilience and fault tolerance.          |
|         Transaction Outbox         | Ensures reliable event publishing.                             |
|         Dual-Write Problem         | Database update succeeds but event publishing fails.           |
|            Saga Pattern            | Coordinates distributed transactions across services.          |
|      Compensating Transactions     | Undo completed actions after a failure.                        |
|          Choreography Saga         | Services coordinate through events.                            |
|         Orchestration Saga         | A coordinator manages the workflow.                            |
|            Kafka + Saga            | Kafka moves events, Saga maintains consistency.                |
|     Transaction Outbox vs Saga     | Outbox ensures delivery; Saga ensures business consistency.    |
|                CQRS                | Separates reads from writes.                                   |
|            CQRS + Kafka            | Kafka updates the read model from write events.                |
|            CQRS vs Saga            | CQRS handles read/write separation; Saga handles transactions. |
|           Event Sourcing           | Stores events instead of current state.                        |
|             Event Store            | Keeps complete event history.                                  |
|        Event Sourcing + CQRS       | Events drive writes; read models handle queries.               |
|        Strangler Fig Pattern       | Gradually replaces a monolith with microservices.              |
|        Incremental Migration       | Migrates one feature at a time.                                |

A **monolith** is a single large application where all components are tightly coupled, while microservices break the application into small, independent services that can be developed, deployed, and scaled separately.

> Microservices are an architectural approach in which a large application is divided into a collection of small, independent services, with each service responsible for a specific business function such as users, orders, payments, or inventory. 

Each microservice has its own codebase, database, and deployment process, allowing teams to develop, scale, update, and maintain services independently without affecting the entire system. 

The services communicate with one another through APIs or messaging systems such as Kafka, creating a flexible and scalable architecture that improves fault isolation, supports continuous delivery, and enables organizations to build complex applications more efficiently.

## API Gateway
> An API Gateway is a centralized entry point for a microservices system that handles request routing, authentication, rate limiting, monitoring, and other cross-cutting concerns while hiding the complexity of the underlying services from clients.

Without an API Gateway
```
Structure                               Getting user details
Client                                  GET /users/123
  ├── Users Service                     GET /orders/user/123
  ├── Orders Service                    GET /points/user/123
  ├── Products Service
  └── Payments Service  
```

With an API Gateway
```
Structure                               Getting user details:
               Client                   GET /dashboard/123
                  │                     
                  ▼                     The gateway calls multiple
           API Gateway                  services internally
         ┌────┼────┬────┐
         ▼    ▼    ▼    ▼
      Users Orders Products Payments    
     Service Service Service Service    
```

## Database per Service
> Database per Service pattern is where each microservice owns and manages its own database.<br>
No other service is allowed to directly access that database.

```
User Service        →   Users DB

Orders Service      →   Orders DB

Payments Service    →   Payments DB
```

- Each service owns its own database
- Teams can work separately
- Easier deployments
- Better fault isolation
- Each service can choose its own database technology



## Event-Driven Architecture (EDA)
> Event-Driven Architecture is a software architecture pattern where services communicate *asyncronously* through events.<br>
Kafka acts as an *event broker* that stores and distributes events from producers to consumers, enabling loosely coupled, scalable, and resilient microservices.
```
           Kafka
             │
             ▼
      OrderCreated Event
             │
   ┌─────────┼─────────┐
   ▼         ▼         ▼
Payment  Inventory  Email
Service   Service   Service
```
The Orders Service creates the event but does not know which services consumes the event.

### Kafka
Apache Kafka is an event streaming platform used as a message broker between services.

Kafka stores events in **Topics**:

Topic: orders

```json
{
    "order_id"  : 101
    ,"user_id"  : 5
    ,"amount"   : 99.99
}

{
    "order_id"  : 102
    ,"user_id"  : 2
    ,"amount"   : 49.99
}
```



### Core concepts
**Producer**<br>
A Producer sends a message to Kafka
```py
producer.send(
    "orders",
    {
        "order_id" : 101
    }
)
```

**Topic**<br>
A category of events.
```
orders
payments
users
inventory
```

**Consumer**<br>
Reads messages from Kafka
```py
for event in consumer:
    print(event)
```


* Loose Coupling - producer does not know the consumers
* Scalability - add consumers without modifying producers
* Reliability - messages remain in Kafka
* Fault Tolerance - consumer downtime does not lose events
* High Throughput - millions of events possible

## Circuit Breaker
> The Circuit Breaker Pattern is a fault-tolerence pattern that moniters service failures.<br>
When failures arise, the pattern temporarily stops requests to an unhealthy dependency, allowing time to recocer while preventing cascading failures and resource exhaustion in a distributed system.

Three states:
```
CLOSED
   │
   ├─ Too many failures
   ▼
OPEN
   │
   ├─ Recovery timeout expires
   ▼
HALF_OPEN
   │
   ├─ Success → CLOSED
   └─ Failure → OPEN
```


**1. Closed**
- Normal operation
- Requests are allowed through
- Failures are tracked

**2. Open**
- Service is considered unhealthy
- Requests are rejected (fail fast)
- No call is made to the dependent service

**3. Half_open**
- After a timeout period, allows a small number of test requests
    - if successful, the circuit closes
    - if unsuccessful, the circuit reopens

- Prevents cascading failures (other services breaking in the pipeline)
- Reduces unnecessary network calls
- Improves application responsiveness
- Gives failing services time to recover
- Increases overall system resilience

## Retry and Bulkhead
Retry Pattern
> Automatically retries failed operations that are likely to succeed on a subsequent attempt due to temprary issues.

It is useful for issues such as:
- Network glitches
- Temporary service outages
- Database 


Bulkhead pattern
> Isolates resources and workloads so that failures or high load in one componenet do not affect the rest of the system.

If one compartment of the ship floods, the rest of ship does not go down with it.

### Retry + Circuit Breaker + Bulkhead
```
   Request
      │
      ▼
    Retry
      │
      ▼
Circuit Breaker
      │
      ▼
   Bulkhead
      │
      ▼
External Service
```
- Retry - handles temporary failures
- Circuir Breaker - stops repeated failed calls
- Bulkhead - Projects the rest of the application

> Retry handles transient failures, Circuit Breaker prevents repeated calls to unhealthy services.<br>
Bulkhead isolates resources so that one failing dependency cannot impact the entire application.<br>
Together they improve the resilience and fault tolerence of a microservice architecture.


## Transaction Outbox
>The Transaction Outbox Pattern ensures reliable event publishing by storing business data changes and outgoing events within the same database transaction. <br>A background process later reads the outbox table and publishes the events to a message broker such as Kafka, preventing inconsistencies between the database and the event stream.

For example:
1. The Order Service creates an order and publishes the event to Kafka
2. The Order is saved to Database BUT the event failed to publish.
3. This creates data inconsistency.

**Transaction Outbox Solution**

Instead of publishing the event to Kafka, both actions occur in *one database transaction*.

In the event that Kafka is down and misses the order event, it can then read the outbox to ensure data consistency.

- Reliable event publishing
- Solves dual-write problems
- Often used with Kafka



## SAGA Pattern
> The Saga Pattern is a distributed transaction pattern that coordinates a series of local transactions across multiple micorservices<br>
If a failure occurs, compensating transactions are executed to undo previously completed actions and maintain data consistency.

In a monolithic system, orders, payments, inventory is done in a single database.<br>
If there are any failures, it will rollback to a previous state.

In microservices, orders, payments, inventory have their respective databases.<br>
A single database transaction cannot span all three databases.

#### Example
A customer places an order:
```
1. Create order         Success
2. Charge payment       Success
3. Reserve inventory    Success
```

Order complete.
#### Break in the system
```
1. Create order         Success
2. Charge payment       Success
3. Reserve inventory    Failed
```
Now the customer has been charged but the order cannot be fulfilled.

Saga can resolve this.
#### Compensating Transactions
Instead of rolling back the database, we can make the following entries in the database:
```
Order Cancelled
Payment Refunded
```

### Choreography Saga
Services communicate through events.

No central coordinator:
```
   Order Service
        │
        ▼
 OrderCreated Event
        │
        ▼
  Payment Service
        │
        ▼
PaymentCompleted Event
        │
        ▼
 Inventory Service
```
<br><br>
If inventory fails:
```
InventoryFailed Event
        │
        ▼
  Payment Service
        │
        ▼
   Refund Payment
        │
        ▼
   Order Service
        │
        ▼
   Cancel Order
```
| Pros | Cons |
| :-: | :-: |
| No central controller | Harder to follow the workflow | 
| Decoupled services | Many events |
| Works natrually with Kafka | |

### Orchestraction Saga
```
          Saga Coordinator
                  │
        ┌─────────┼─────────┐
        ▼         ▼         ▼
     Order     Payment   Inventory
```

Flow goes:
```
Create Order → Charge Payment → Reserve Inventory
```
<br><br>
If inventory fails:
```
No Inventory → Refund Payment → Cancel Order
```


| Pros | Cons |
| :-: | :-: |
| Easier to understand | Central coordinator becomes important | 
| Easier monitoring | |

### Relationship to Kafka
A common pattern is:
```
Transaction Outbox
        │
        ▼
      Kafka
        │
        ▼
       Saga
```
<br><br>
Example:
```
OrderCreated Event
        │
        ▼
Kafka Topic
        │
        ▼
Payment Service
```

Kafka transports the events.

Saga manages business consistency.

### Transaction Outbox vs. Saga
Transaction Outbox focuses on parity between Database and Event Consistency.<br>
I.e. Was the event published reliably?

Saga focuses on Business Transaction Consistency.<br>
I.e. What happens when one service fails halfway through a workflow?

## Command Query Responsibility Segregation (CQRS)
> CQRS is an architectural pattern that separates operations that modify data (commands) from operations that read data (queries), often using different models or databases optimised for their respective purposes.

The idea is:<br>
<div style="text-align: center;">
Use one model for writing data, use another model for reading data
</div>
<br><br>
Without CQRS, one service handles both reads and writes.<br>
Everything uses the same database.

However with CQRS, services are split into commands and queries.
```
        Commands (Write)
                │
                ▼
          Write Database

                │
           Events/Kafka
                │

                ▼
          Read Database
                ▲
                │
          Queries (Read)


### Why use CQRS?
If we have 10 writes/sec and 1000 reads/sec - majority of traffic is reads.

* Without CQRS - one database handles everything
* With CQRS - with a split database, we can scale things independently.

### CQRS + Kafka
Command → Write DB → OrderCreate Event → Kafka → Read DB Updated → Queries

### Relationship to Saga
Saga focuses on Business Transactions

CQRS focuses on read/write separation.


| Pros | Cons |
| :-: | :-: |
| Faster reads | Added complexity | 
| Independent scaling | There will be lag between reading updated data |
| Better reporting | |
| Works well with Kafka | |
| Works well with Event Sourcing| 


## Event Sourcing
> Event Sourcing is an architectural pattern where state changes are stored as a sequence of immutable events.<br>
The current state of the system is reconstructed by replaying those events rather than reading the latest row from a database.

Normal databases store the current state:
```
Orders
----------------
ID  Status
1   SHIPPED
```
No details of when it was created, when payment happened, nor when it was processed.
<br><br>
Event Sourcing stores history of events:
```
Events
----------------
1   OrderCreated
2   PaymentReceived
3   OrderProcessed
4   OrderShipped
```
To determine current order status, it is derived from event history.


### Bank Account
Tradtional database would show:
```
Account
----------------
Balance      750
```
<br><br>
With Event Sourcing
```
Account
----------------
Deposit     1000
Withdraw     200
Withdraw      50
```
And the balance reconstructed would therefore be 750.


### Event Sourcing + CRQS
Often paired together as CQRS separates reads and writes and Event Sourcing stores events instead of current state.

**Write Side**<br>
Store events:
- OrderCreated
- PaymentReceived
- OrderShipped

**Read Side**<br>
Build a read model:
- Order Dashboard
- Customer History
- Reports

```
Command → Event Store → Events → Read Database → Queries
```

| Pros | Cons |
| :-: | :-: |
| Complete audit trail | Added complexity | 
| Easy debugging | Event design matters |
| Ability to rebuild state | Replaying can be slow |
| Intergration with Kafka | |


## Strangler Fig Pattern
The name is reference to the strangler fig tree which grows around another tree and slowly replaces it.

> The Strangler Fig is an incremental migration pattern where functionality is moved from a monolithic structure to new services.<br>
Traffic is progressively redirected to the new services until the monolith is completely replaced.

If you have a monolithic architecture and want to move to a microservices based architecture, you could re-write everything all at once then deploy it.

The issues with that deployment are:
- Long project
- High risk
- Difficult to rollback


### Strangler Fig approach
Instead of replacing all features at once, replace one feature at a time.<br>
A case study for this is *Netflix*.

| Pros | Cons |
| :-: | :-: |
| Lower risk | Temporary complexity | 
| Easier testing | Duplicated logic |
| Faster delivery | Routing layer required |
| Rollback friendly | |
